In [20]:
from langgraph.graph import START, END, StateGraph
from langgraph.types import Send
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from typing import Annotated, List, TypedDict
from pydantic import BaseModel, Field
from pathlib import Path
from dotenv import load_dotenv
import operator

In [21]:
load_dotenv()

True

In [22]:
class Task(BaseModel):
    id: int
    title: str
    brief: str = Field(...,description="What to cover")

In [23]:
class Plan(BaseModel):
    blog_title: str
    tasks: List[Task]

In [24]:
class blogState(TypedDict):
    topic: str
    plan: Plan
    sections: Annotated[List[str],operator.add]
    final: str

In [25]:
llm = ChatOpenAI(model='gpt-4.1-mini')

In [44]:
def orchestrator(state:blogState):
    plan = llm.with_structured_output(Plan).invoke(
        [SystemMessage(content="Create a blog plan with 5-7 sections on the following topic. The title should be the name of a markdown file, so use characters accordingly."),
         HumanMessage(content=f"Topic: {state["topic"]}")]
    )
    return {"plan":plan}

In [45]:
def fanout(state:blogState):
    return [Send("worker",{"task":task,"topic":state["topic"],"plan":state["plan"]}) for task in state["plan"].tasks]

In [46]:
def worker(payout:dict)->dict:
    task = payout['task']
    topic = payout['topic']
    plan = payout['plan']

    blog_title = plan.blog_title

    section_md = llm.invoke(
        [SystemMessage(content="Write one clean Markdown section"),
         HumanMessage(content=(
             f"Blog: {blog_title}\n"
             f"Topic: {topic}\n"
             f"Section: {task.title}\n"
             f"Brief: {task.brief}\n\n"
             "Return only the section content in Markdown"
         ))]
    ).content.strip()

    return {"sections":[section_md]}

In [47]:
def reducer(state:blogState):
    title = state['plan'].blog_title
    body = "\n\n".join(state['sections']).strip()

    final_md = f"# {title}\n\n{body}\n"

    filename = title.lower().replace(" ","_") + ".md"
    print(filename)
    output_path = Path(filename)
    output_path.write_text(final_md,encoding="utf-8")

    return {"final":final_md}

In [48]:
graph = StateGraph(blogState)

graph.add_node("orchestrator",orchestrator)
graph.add_node("worker",worker)
graph.add_node("reducer",reducer)

graph.add_edge(START,"orchestrator")
graph.add_conditional_edges("orchestrator",fanout,["worker"])
graph.add_edge("worker","reducer")
graph.add_edge("reducer",END)

app = graph.compile()

In [49]:
out = app.invoke({"topic":"Write a blog on Self Attention"})

understanding_self_attention_in_deep_learning.md


In [37]:
print(out['sections'])

['## Introduction to Self-Attention\n\nSelf-attention is a powerful mechanism in deep learning that allows a model to weigh the importance of different parts of a single input sequence when making predictions. Unlike traditional neural networks that process input data uniformly or rely heavily on fixed-length context windows, self-attention dynamically adjusts how much focus to allocate to each element relative to others within the same sequence.\n\nIn natural language processing (NLP), this capability is crucial because the meaning of a word often depends on its context. For example, the word "bank" can refer to a financial institution or the side of a river, and determining the correct interpretation requires understanding surrounding words. Self-attention enables models to capture these nuanced relationships by computing attention scores that represent the relevance between any two words in a sentence, regardless of their distance from each other.\n\nThis mechanism forms the backbon